In [2]:
import os
import sqlite3

In [3]:
connection = sqlite3.connect('mydb.db')

In [4]:
connection

In [5]:
table_creation_query1 = """
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER NOT NULL PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    hire_date TEXT NOT NULL,
    salary REAL NOT NULL
);
"""

In [6]:
table_creation_query2 = """
CREATE TABLE IF NOT EXISTS customers (
    id INTEGER NOT NULL PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT
);
"""

In [7]:
table_creation_query3 = """
CREATE TABLE IF NOT EXISTS orders (
    id INTEGER NOT NULL PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers (id)
);
"""

In [8]:
cursor = connection.cursor()

In [9]:
cursor.execute(table_creation_query1)
cursor.execute(table_creation_query2)
cursor.execute(table_creation_query3)

In [10]:
employee_data = [
    (1, 'John', 'Doe', 'john.doe@company.com', '2020-01-15', 75000.00),
    (2, 'Jane', 'Smith', 'jane.smith@company.com', '2019-03-20', 82000.00),
    (3, 'Robert', 'Johnson', 'robert.johnson@company.com', '2021-06-10', 68000.00),
    (4, 'Maria', 'Garcia', 'maria.garcia@company.com', '2020-11-01', 71000.00),
    (5, 'David', 'Williams', 'david.williams@company.com', '2018-09-05', 95000.00)
]

cursor.executemany("""
    INSERT OR IGNORE INTO employees (id, first_name, last_name, email, hire_date, salary)
    VALUES (?, ?, ?, ?, ?, ?)
""", employee_data)

customer_data = [
    (1, 'Alice', 'Brown', 'alice.brown@email.com', '555-0101'),
    (2, 'Bob', 'Miller', 'bob.miller@email.com', '555-0102'),
    (3, 'Carol', 'Davis', 'carol.davis@email.com', '555-0103'),
    (4, 'Eve', 'Wilson', 'eve.wilson@email.com', None),
    (5, 'Frank', 'Moore', 'frank.moore@email.com', '555-0105')
]

cursor.executemany("""
    INSERT OR IGNORE INTO customers (id, first_name, last_name, email, phone)
    VALUES (?, ?, ?, ?, ?)
""", customer_data)

order_data = [
    (1, 1, '2023-01-15', 249.99),
    (2, 1, '2023-02-20', 499.50),
    (3, 2, '2023-01-28', 125.75),
    (4, 3, '2023-03-05', 899.00),
    (5, 2, '2023-03-12', 350.25),
    (6, 4, '2023-02-14', 175.50),
    (7, 5, '2023-01-30', 675.00),
    (8, 3, '2023-03-18', 220.80),
    (9, 1, '2023-03-22', 129.99),
    (10, 5, '2023-02-28', 450.00)
]

cursor.executemany("""
    INSERT OR IGNORE INTO orders (id, customer_id, order_date, amount)
    VALUES (?, ?, ?, ?)
""", order_data)

connection.commit()

In [11]:
cursor.execute("select * from customers")

In [12]:
for row in cursor.fetchall():
    print(row)

(1, 'Alice', 'Brown', 'alice.brown@email.com', '555-0101')
(2, 'Bob', 'Miller', 'bob.miller@email.com', '555-0102')
(3, 'Carol', 'Davis', 'carol.davis@email.com', '555-0103')
(4, 'Eve', 'Wilson', 'eve.wilson@email.com', None)
(5, 'Frank', 'Moore', 'frank.moore@email.com', '555-0105')


In [13]:
from langchain_community.utilities import SQLDatabase

C:\Users\tagir\AppData\Local\Temp\ipykernel_11360\1266060898.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [14]:
db = SQLDatabase.from_uri("sqlite:///mydb.db")

In [15]:
db.dialect

'sqlite'

In [16]:
db.get_usable_table_names()

['customers', 'employees', 'orders']

In [17]:
import os
os.environ["GROQ_API_KEY"] = ".."

In [18]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b")

In [19]:
llm.invoke("Does Groq charge money?")

AIMessage(content='**Short answer:**  \nYes – Groq’s services are not free. They charge for the use of their hardware and cloud platform, though they do offer a limited free tier for developers to experiment with.\n\n---\n\n### How Groq’s pricing works\n\n| Service | Pricing model | Notes |\n|---------|---------------|-------|\n| **Hardware (on‑prem or embedded)** | **Capital expense** (purchase price) | You buy a Groq accelerator and host it yourself. The cost depends on the model (e.g., Groq\u202f1, Groq\u202f2, etc.) and the number of units. |\n| **Cloud (Groq Cloud)** | **Pay‑as‑you‑go** (per‑second billing) | You spin up a virtual “Groq instance” and are billed for the time the instance is running. The price per second varies by instance size and region. |\n| **Developer/Trial tier** | **Free tier** | Groq offers a small, free tier (e.g., a few hours per month) to let developers prototype and test workloads. Once you exceed the free quota, you’re switched to the normal pay‑as‑you‑

In [20]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [21]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [22]:
tools = toolkit.get_tools()

In [23]:
for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [24]:
list_tables_tool = next((tool for tool in tools if tool.name == 'sql_db_list_tables'), None)

In [25]:
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001F339ECFDA0>)

In [28]:
get_shema_tool = next((tool for tool in tools if tool.name == 'sql_db_schema'), None)

In [29]:
get_shema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001F339ECFDA0>)

In [30]:
list_tables_tool.invoke("")

'customers, employees, orders'

In [34]:
print(get_shema_tool.invoke("customers, employees, orders"))


CREATE TABLE customers (
	id INTEGER NOT NULL, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
id	first_name	last_name	email	phone
1	Alice	Brown	alice.brown@email.com	555-0101
2	Bob	Miller	bob.miller@email.com	555-0102
3	Carol	Davis	carol.davis@email.com	555-0103
*/


CREATE TABLE employees (
	id INTEGER NOT NULL, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	hire_date TEXT NOT NULL, 
	salary REAL NOT NULL, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)

/*
3 rows from employees table:
id	first_name	last_name	email	hire_date	salary
1	John	Doe	john.doe@company.com	2020-01-15	75000.0
2	Jane	Smith	jane.smith@company.com	2019-03-20	82000.0
3	Robert	Johnson	robert.johnson@company.com	2021-06-10	68000.0
*/


CREATE TABLE orders (
	id INTEGER NOT NULL, 
	customer_id INTEGER NOT NULL, 
	order_date TEXT NOT NULL, 
	amount REAL NOT NULL, 
	PRIMARY KEY (id